# Day 3 · Exercise 5: How much should we order when demand is uncertain?

**What you'll be able to say by the end**
1. "When a lost sale costs 3× more than a leftover, the best order is the **75th percentile** of the demand forecast, not the mean."
2. "Across 50 real Walmart items over 52 weeks, ordering the 75% quantile instead of the mean cut cost by **20%**. Ordering the mean ran out of stock in **47% of weeks**."
3. "The newsvendor cost is exactly the **pinball loss** at τ = critical ratio. Training a quantile model on pinball loss *is* optimizing the inventory decision."
4. "The further the critical ratio is from 0.5, the more a quantile forecast is worth."

**This is the bridge from your forecasting work to an inventory decision.**

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


## 1. The story

You have to decide **how many units to order before you know demand**. Shoes for a season, newspapers for a day, fresh food for a week: the order is placed, then demand shows up, and you live with the mismatch.

Two ways to be wrong:
- **Order too many:** each leftover costs you what you paid minus what you get back clearing it. That's the **overage cost, co**.
- **Order too few:** each customer you turn away costs you the margin you didn't make. That's the **underage cost, cu**.

We'll use **Nike-like margins**: a unit costs us 40% of its price, and a leftover is cleared at 20% of its price. So:
- cu = price − cost = 60% of the price
- co = cost − clearance value = 20% of the price

**Running short costs 3× more than a leftover.** Keep that in mind: it's the whole answer.

**Data:** weekly sales of 50 items in Walmart store CA_1 from the M5 dataset, built by `data/prepare_m5.py`. We treat each week as one ordering decision.

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd().parent  # notebooks/ is the working folder
OUT = ROOT / "outputs"
weekly = pd.read_csv(ROOT / "data" / "processed" / "m5_ca1_weekly.csv")

# Margins, as a share of the selling price (Nike-like: high margin, some markdown value)
COST_SHARE = 0.40     # we pay 40% of the price for each unit
SALVAGE_SHARE = 0.20  # a leftover unit is cleared at 20% of the price
HISTORY_WEEKS = 26    # the "forecast" looks at the last 26 weeks
TEST_WEEKS = 52       # backtest on the last 52 weeks

units = weekly.pivot(index="wm_yr_wk", columns="item_id", values="units")
price = weekly.pivot(index="wm_yr_wk", columns="item_id", values="price")
WEEKS = list(units.index)
ITEMS = list(units.columns)
category = weekly.drop_duplicates("item_id").set_index("item_id")["cat_id"]

print(f"{len(ITEMS)} items, {len(WEEKS)} weeks, categories: {category.value_counts().to_dict()}")
weekly.head()

50 items, 277 weeks, categories: {'FOODS': 20, 'HOBBIES': 15, 'HOUSEHOLD': 15}


,item_id,cat_id,wm_yr_wk,week_start,units,price
0,FOODS_1_154,FOODS,11101,2011-01-29,17,1.25
1,FOODS_1_154,FOODS,11102,2011-02-05,16,1.25
2,FOODS_1_154,FOODS,11103,2011-02-12,4,1.25
3,FOODS_1_154,FOODS,11104,2011-02-19,11,1.25
4,FOODS_1_154,FOODS,11105,2011-02-26,11,1.25


## 2. One item by hand

Take one item, `HOUSEHOLD_1_517`, and pretend its last 26 weeks of sales are our picture of what next week might look like: 26 equally likely outcomes. (That's a crude forecast. Section 4 explains where your real model fits in.)

For every possible order size, work out the average cost over those 26 outcomes.

In [3]:
def costs_per_unit(p, cost_share=COST_SHARE, salvage_share=SALVAGE_SHARE):
    """Underage cost cu (a lost sale loses the margin) and overage cost co (a leftover loses cost - salvage)."""
    cu = p * (1 - cost_share)
    co = p * (cost_share - salvage_share)
    return cu, co


def newsvendor_cost(order, demand, cu, co):
    """Cost of ordering `order` when `demand` turns up: co per leftover unit, cu per unit short."""
    return co * np.maximum(order - demand, 0) + cu * np.maximum(demand - order, 0)


item = "HOUSEHOLD_1_517"
last = WEEKS[-TEST_WEEKS - 1]
history = units.loc[WEEKS[WEEKS.index(last) - HISTORY_WEEKS + 1:WEEKS.index(last) + 1], item]
p = price.loc[last, item]
cu, co = costs_per_unit(p)
print(f"price ${p:.2f}: cu = ${cu:.2f} per unit short, co = ${co:.2f} per unit left over")
print("26 possible demands, sorted:", sorted(history.astype(int).tolist()))

tries = pd.DataFrame([{"order": q, "avg cost $": newsvendor_cost(q, history, cu, co).mean()}
                      for q in range(int(history.min()), int(history.max()) + 1)])
best = tries.loc[tries["avg cost $"].idxmin()]
mean_q = round(history.mean())

fig = go.Figure(go.Scatter(x=tries["order"], y=tries["avg cost $"], mode="lines+markers",
                           hovertemplate="order %{x}<br>avg cost $%{y:.2f}<extra></extra>"))
fig.add_vline(x=mean_q, line_dash="dot", line_color="gray", annotation_text=f"mean = {mean_q}")
fig.add_vline(x=best["order"], line_dash="dot", line_color="#E45756", annotation_text=f"best = {best['order']:.0f}",
              annotation_position="top left")
fig.update_layout(title=f"{item}: average weekly cost for each order size", xaxis_title="units ordered",
                  yaxis_title="average cost per week ($)", height=420, width=900, template="plotly_white")
fig.show()
print(f"Best order: {best['order']:.0f} units, ${best['avg cost $']:.2f}/week. "
      f"Ordering the mean ({mean_q}) costs ${newsvendor_cost(mean_q, history, cu, co).mean():.2f}/week.")

price $0.98: cu = $0.59 per unit short, co = $0.20 per unit left over
26 possible demands, sorted: [28, 31, 39, 41, 42, 45, 46, 48, 50, 52, 52, 55, 57, 58, 59, 60, 60, 62, 65, 66, 66, 67, 69, 72, 74, 78]


Best order: 66 units, $2.97/week. Ordering the mean (55) costs $4.22/week.


**The best order is 66, well above the mean of 55**, and it costs about 30% less ($2.97 vs $4.22 a week).

The curve is lopsided. To the left of the best point (ordering too few) it climbs steeply, because each missing unit costs $0.59. To the right (ordering too many) it climbs gently, because each extra unit costs only $0.20. When being short hurts more, you lean towards ordering more.

## 3. The critical ratio: the rule behind the curve

Think about the **last unit** you're deciding whether to add:
- It **helps** if demand turns out higher than your order: you avoid losing **cu**. That happens with probability P(demand > Q).
- It **hurts** if demand turns out lower: you waste **co**. That happens with probability P(demand ≤ Q).

Keep adding units while the expected help beats the expected harm. Stop where they balance:

> cu × P(demand > Q) = co × P(demand ≤ Q)

Rearranging gives the **newsvendor rule**: order the quantity Q where

```latex
P(\text{demand} \le Q) = \frac{c_u}{c_u + c_o}
```

That fraction is the **critical ratio**. **The best order is the critical-ratio quantile of the demand forecast.**

In [4]:
cr = cu / (cu + co)
print(f"critical ratio = cu / (cu + co) = {cu:.2f} / ({cu:.2f} + {co:.2f}) = {cr:.2f}")
print(f"{cr:.0%} quantile of the 26 possible demands: {history.quantile(cr):.1f}")
print(f"Best order found by trying every size:        {best['order']:.0f}")

critical ratio = cu / (cu + co) = 0.59 / (0.59 + 0.20) = 0.75
75% quantile of the 26 possible demands: 65.8
Best order found by trying every size:        66


Same answer, with no trial and error. Here cu is 3× co, so the critical ratio is 3 ÷ (3 + 1) = **0.75**: order enough to cover demand in 75% of weeks, and accept running short in the other 25%.

**How to read it in an interview:**
- Critical ratio **above 0.5** (high margin, cheap leftovers: fashion, Nike launches): order **above** the median.
- Critical ratio **below 0.5** (low margin, costly leftovers: perishables, tech that goes obsolete): order **below** the median.
- **Ordering the mean is only right when the ratio is about 0.5** and demand is symmetric. That's rarely true.

## 4. The link to your forecasting work: pinball loss

Quantile forecasting models (LightGBM with `objective="quantile"`, TimesFM's quantile heads) are trained and scored on the **pinball loss**:

```latex
L_\tau(\hat q, y) = \begin{cases} \tau\,(y - \hat q) & y \ge \hat q \\ (1-\tau)\,(\hat q - y) & y < \hat q \end{cases}
```

Compare it with the newsvendor cost: **cu × units short + co × units left over**. Set τ = cu ÷ (cu + co), and the newsvendor cost is exactly **(cu + co) × pinball loss**. Let's check:

In [5]:
def pinball_loss(forecast, actual, tau):
    """The quantile (pinball) loss used to train and score quantile forecasts."""
    return np.where(actual >= forecast, tau * (actual - forecast), (1 - tau) * (forecast - actual))


q = history.quantile(cr)
print(f"average newsvendor cost:          ${newsvendor_cost(q, history, cu, co).mean():.4f}")
print(f"(cu + co) x average pinball loss: ${(cu + co) * pinball_loss(q, history, cr).mean():.4f}")

average newsvendor cost:          $2.9739
(cu + co) x average pinball loss: $2.9739


**They're identical.** So when you train a quantile model at τ = 0.75 by minimizing pinball loss, you're directly minimizing this inventory cost. **Your forecast and the inventory decision are the same optimization.**

That's your bridge story for the interview: *"A point forecast tells you the middle. The business decision needs a specific quantile, set by the cost of being short versus being long. Quantile forecasts trained on pinball loss give you that decision directly."*

**In this notebook the "forecast" is simple on purpose:** the mean, or the chosen quantile, of the last 26 weeks. Your LightGBM or TimesFM quantile output drops straight into these two functions:

In [6]:
# The "forecast": a stand-in for your LightGBM / TimesFM model. Point forecast = mean of the
# last 26 weeks; quantile forecast = that quantile of the last 26 weeks. Swap in real model output here.
def forecast_mean(history):
    return history.mean()


def forecast_quantile(history, q):
    return history.quantile(q)

## 5. Does it hold up on real data? 50 items × 52 weeks

**Backtest:** for each of the last 52 weeks and each of the 50 items, forecast from the 26 weeks before, place the order two ways (the mean, or the 75% quantile), then see what demand actually did.

In [7]:
def backtest(critical_ratio=None, cost_share=COST_SHARE, salvage_share=SALVAGE_SHARE):
    """Order each of the 50 items every week for the last 52 weeks, two ways. Returns one row per item-week."""
    rows = []
    for t in WEEKS[-TEST_WEEKS:]:
        i = WEEKS.index(t)
        history = units.loc[WEEKS[i - HISTORY_WEEKS:i]]
        demand = units.loc[t]
        cu, co = costs_per_unit(price.loc[t], cost_share, salvage_share)
        if critical_ratio is not None:  # override margins: keep cu + co = price, set the ratio directly
            cu, co = price.loc[t] * critical_ratio, price.loc[t] * (1 - critical_ratio)
        cr = float((cu / (cu + co)).iloc[0])
        for plan, order in [("order the mean", forecast_mean(history)),
                            (f"order the {cr:.0%} quantile", forecast_quantile(history, cr))]:
            order = order.round()
            rows.append(pd.DataFrame({
                "plan": plan, "week": t, "item": ITEMS, "category": category[ITEMS].to_numpy(),
                "order": order.to_numpy(), "demand": demand.to_numpy(),
                "cost": newsvendor_cost(order, demand, cu, co).to_numpy(),
                "sold": np.minimum(order, demand).to_numpy(),
                "leftover": np.maximum(order - demand, 0).to_numpy(),
                "ran out": (demand > order).to_numpy(),
            }))
    return pd.concat(rows, ignore_index=True)


def summarize(results):
    g = results.groupby("plan", sort=False).agg(
        cost=("cost", "sum"), sold=("sold", "sum"), demand=("demand", "sum"),
        leftover_units=("leftover", "sum"), weeks_ran_out=("ran out", "mean"))
    g["fill rate"] = g["sold"] / g["demand"]
    return g[["cost", "fill rate", "weeks_ran_out", "leftover_units"]]


results = backtest()
summary = summarize(results)
saving = 1 - summary["cost"].iloc[1] / summary["cost"].iloc[0]
print(f"Ordering the 75% quantile instead of the mean cuts cost by {saving:.1%}")
summary.assign(**{"cost": summary["cost"].map("${:,.0f}".format),
                  "fill rate": summary["fill rate"].map("{:.1%}".format),
                  "weeks_ran_out": summary["weeks_ran_out"].map("{:.1%}".format)})

Ordering the 75% quantile instead of the mean cuts cost by 20.4%


,cost,fill rate,weeks_ran_out,leftover_units
plan,,,,
order the mean,"$22,248",85.9%,46.8%,11105.0
order the 75% quantile,"$17,705",93.2%,25.8%,21538.0


In [8]:
by_item = results.groupby(["item", "plan"], sort=False)["cost"].sum().unstack()
by_item.columns = ["mean", "quantile"]
by_item["category"] = category[by_item.index]

fig = go.Figure()
for cat, g in by_item.groupby("category"):
    fig.add_trace(go.Scatter(x=g["mean"], y=g["quantile"], mode="markers", name=cat, text=g.index,
                             hovertemplate="%{text}<br>mean plan $%{x:,.0f}<br>quantile plan $%{y:,.0f}<extra></extra>"))
top = by_item[["mean", "quantile"]].to_numpy().max()
fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines", line=dict(dash="dot", color="gray"), name="same cost"))
fig.update_layout(title="Yearly cost per item: below the dotted line, the quantile plan is cheaper",
                  xaxis_title="cost ordering the mean ($/year)", yaxis_title="cost ordering the 75% quantile ($/year)",
                  height=500, width=700, template="plotly_white")
fig.show()
print(f"Quantile plan is cheaper for {(by_item['quantile'] < by_item['mean']).sum()} of {len(by_item)} items")

Quantile plan is cheaper for 45 of 50 items


**What happened:**
- **Ordering the mean ran out of stock in 47% of item-weeks.** That makes sense: if demand is roughly symmetric, it lands above its mean about half the time. In a high-margin business that's a lot of lost sales.
- **Ordering the 75% quantile** ran out in 26% of weeks, close to the 25% it was designed for. Fill rate went from 86% to 93%.
- It **leaves more units over** (about 21,500 vs 11,100), and that's deliberate. A leftover costs $0.20 on the dollar, and a lost sale costs $0.60.
- **Total cost fell 20%**, and it fell in every category: FOODS 20%, HOBBIES 20%, HOUSEHOLD 22%.

The critical ratio isn't a theory result that falls apart on real data. A crude 26-week empirical quantile already captures most of the gain. A better quantile model would do better still.

## 6. Change the margins: when is the quantile worth the most?

The 75% case depended on our margin assumption. Let's re-run the whole backtest for critical ratios from 0.10 (leftovers very expensive) to 0.95 (lost sales very expensive).

In [9]:
rows = []
for ratio in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95]:
    s = summarize(backtest(critical_ratio=ratio))
    rows.append({"critical ratio": ratio, "saving vs mean": 1 - s["cost"].iloc[1] / s["cost"].iloc[0],
                 "fill rate (quantile)": s["fill rate"].iloc[1], "fill rate (mean)": s["fill rate"].iloc[0]})
sweep = pd.DataFrame(rows)

fig = go.Figure(go.Scatter(x=sweep["critical ratio"], y=100 * sweep["saving vs mean"], mode="lines+markers",
                           hovertemplate="critical ratio %{x}<br>saving %{y:.1f}%<extra></extra>"))
fig.add_hline(y=0, line_color="gray")
fig.add_vline(x=0.75, line_dash="dot", line_color="gray", annotation_text="our margins")
fig.update_layout(title="Cost saved by ordering the critical-ratio quantile instead of the mean",
                  xaxis_title="critical ratio cu / (cu + co)", yaxis_title="cost saved (%)",
                  height=430, width=900, template="plotly_white")
fig.show()
sweep.round(3)

,critical ratio,saving vs mean,fill rate (quantile),fill rate (mean)
0,0.10,0.539,0.616,0.859
1,0.20,0.277,0.698,0.859
2,0.30,0.104,0.756,0.859
3,0.40,0.018,0.806,0.859
4,0.50,-0.007,0.846,0.859
5,0.60,0.028,0.883,0.859
6,0.70,0.124,0.915,0.859
7,0.75,0.204,0.932,0.859
8,0.80,0.291,0.945,0.859
9,0.90,0.541,0.969,0.859


**A V shape:**
- **Near 0.5, no gain.** The 50% quantile (the median) is close to the mean, so both plans order about the same. (At exactly 0.5 the mean is even slightly better here: 26 weeks of history is a noisy estimate of the median.)
- **Far from 0.5, big gains:** about 54% at 0.1 and 0.9, and 71% at 0.95. The mean is the wrong target, and the further the ratio is from 0.5, the more wrong it is.
- **Fill rate moves with the ratio**, from 62% at 0.1 to 98% at 0.95. Ordering the mean gives 86% whatever the margins are. That's the real problem with a point forecast: it can't respond to the economics.

**Interview line:** *"How much a quantile forecast is worth depends on the cost asymmetry. For a high-margin brand like Nike, where a lost sale costs several times a markdown, it's worth a lot."*

## Recap: say these out loud

1. **What's the newsvendor problem?** Choose an order before demand is known, trading the cost of leftovers (co) against the cost of lost sales (cu).
2. **What's the answer?** Order the **critical-ratio quantile** of the demand forecast: P(demand ≤ Q) = cu ÷ (cu + co).
3. **Derive it:** the last unit helps with probability P(D > Q) × cu and hurts with probability P(D ≤ Q) × co. Stop where they're equal.
4. **What happens when shortage cost rises?** The critical ratio rises, so you order a higher quantile: more stock, higher fill rate.
5. **How does it connect to forecasting?** Newsvendor cost = (cu + co) × pinball loss at τ = critical ratio. A quantile model trained on pinball loss optimizes the decision directly.
6. **The result on real data:** 20% lower cost than ordering the mean, across 50 M5 items × 52 weeks. Ordering the mean ran out of stock 47% of the time.

**Words from today**

| Word | Meaning |
|---|---|
| Newsvendor | Single-period order decision under uncertain demand |
| Underage cost, cu | Cost per unit short: the lost margin (and any goodwill) |
| Overage cost, co | Cost per unit left over: purchase cost minus clearance value |
| Critical ratio | cu ÷ (cu + co): the quantile to order at |
| Quantile forecast | A forecast of a specific percentile of demand, not just its middle |
| Pinball (quantile) loss | The loss used to train quantile forecasts; equals newsvendor cost up to a constant |
| Fill rate | Share of demand served from stock |
| Backtest | Replaying past weeks: decide with only the data available then, score on what actually happened |

**Try it yourself:**
1. Set `SALVAGE_SHARE = 0.0` (leftovers are worthless). *Predict:* does the critical ratio go up or down? Re-run section 5.
2. Change `HISTORY_WEEKS` to 8. *Predict:* does the quantile plan get better or worse? Shorter history reacts faster but estimates the 75th percentile from only 8 points.